# ViFeedback — H7 on Kaggle: Qwen3-4B as an LLM reference

The declared H7 run (`configs/experiments/cycle2.yaml`): an instruction-tuned LLM, scored by **label
likelihood** with the prompt frozen on a train subset, against the fine-tuned PhoBERT encoders on the
frozen challenge set and on validation. The laptop ran the Qwen3-1.7B pilot; Qwen3-4B needs this GPU.
`cycle3.yaml` v6 adds one zero-shot run on **NEU-ESC** test (6,613 real student posts), next to the
gpt-4o-mini run already done there: a local model sends no text anywhere.

**Settings (free tier)**

1. *Accelerator → **GPU T4 x2***. Both GPUs are used: the seven configurations run as two parallel
   queues, balanced by workload. Not P100: a single GPU, and recent PyTorch builds drop it.
2. *Internet → **On*** (clones the code from GitHub; downloads UIT-VSFC, NEU-ESC and Qwen3-4B, about
   8 GB).
3. *Add-ons → Secrets → `HF_TOKEN`*: a Hugging Face token of the account that accepted NEU-ESC's
   conditions on its dataset page. Without it only the NEU-ESC configuration is skipped, with a
   message; the other six still run. No API key and no dataset upload are needed.

**Run it in the background:** *Save Version → Save & Run All (Commit)*. The browser can be closed and
the output (`h7_results.zip`) is kept with the version. Expected: about 5 min of setup, 2 min of sanity
check, then roughly 15–30 min for the seven configurations on two T4s: well inside the weekly 30
GPU-hour quota. An interactive *Run All* works too; run it again in the same session after an
interruption and every configuration that already finished is skipped.

**What the version keeps:** only `h7_results.zip` (labels, probabilities and logs). The code and the
corpora live in `/tmp/vifeedback`, which Kaggle does not save, so no UIT-VSFC or NEU-ESC text ends up
in the notebook's output.

Memory and speed:
- fp16 weights, about 8 GB per GPU, loaded **once per GPU** for its whole queue.
- The system prompt and demonstrations are the same for every prompt of a configuration (82–95% of
  its tokens). They are run **once**, and their key/value cache is reused. A self-check against full
  scoring comes first; if it disagrees, full scoring is used, and the summary records which.
  In a laptop test with Qwen3-0.6B this was 4–5× faster on UIT-VSFC prompts. It was also no further
  from one-prompt-at-a-time scoring than batched full scoring is.
- Prompts are sorted by length and batched by a token budget; logits are computed for the last
  position only.

Every cell stops loudly on an error.

**Bring the results home.** Download `h7_results.zip` from the *Output* panel, extract it into its own
folder on the laptop (e.g. `kaggle_results/`, never over the repository), then:

```
vifeedback results merge kaggle_results --dry-run
vifeedback results merge kaggle_results
vifeedback study h7-decide
```


In [ ]:
# Configuration. The defaults are the declared run; do not change them for the real run.
import os
import pathlib

REPO_URL = os.environ.get('H7_REPO', 'https://github.com/ThanhDatVN/ViFeedback-NLP-Service.git')
BRANCH = os.environ.get('H7_BRANCH', 'main')
MODEL = os.environ.get('H7_MODEL', 'Qwen/Qwen3-4B')          # declared model (cycle2.yaml H7)
DTYPE = os.environ.get('H7_DTYPE', 'float16')               # T4 has no fast bfloat16
BATCH_SIZE = int(os.environ.get('H7_BATCH', '64'))          # prompts per batch; the token budget usually binds first
MAX_TOKENS = int(os.environ.get('H7_MAX_TOKENS', '12288'))  # padded tokens per batch; fits a 15 GiB T4
SANITY_MIN_ACC = float(os.environ.get('H7_SANITY_MIN', '0.45'))
# (data, shots, demonstration seed): zero-shot, and 6-shot with the two declared draws (cycle2.yaml);
# NEU-ESC zero-shot (cycle3.yaml v6).
CONFIGS = [
    (d, int(k), int(s))
    for d, k, s in (c.split(':') for c in os.environ.get(
        'H7_CONFIGS',
        'challenge:0:1,challenge:6:1,challenge:6:2,validation:0:1,validation:6:1,validation:6:2,neu_esc:0:1',
    ).split(','))
]
ON_KAGGLE = pathlib.Path('/kaggle/working').exists()
# The zip of results goes to OUT_BASE, which the version keeps. The code and the corpora go to /tmp,
# which it does not, so no UIT-VSFC or NEU-ESC text is saved with the notebook's output.
OUT_BASE = pathlib.Path('/kaggle/working') if ON_KAGGLE else pathlib.Path(os.environ.get('H7_WORKDIR', '.')).resolve()
WORK = (pathlib.Path('/tmp') if ON_KAGGLE else OUT_BASE) / 'vifeedback'
print(f'model {MODEL} | dtype {DTYPE} | batch <= {BATCH_SIZE} prompts and <= {MAX_TOKENS} tokens | '
      f'{len(CONFIGS)} configurations | work dir {WORK}')

In [ ]:
# Environment: GPUs, internet, the Hugging Face token (needed for NEU-ESC only).
import shutil
import socket
import subprocess

import torch

assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4 x2, then run again.'
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory / 2**30:.1f} GiB, capability {p.major}.{p.minor}')
print('torch', torch.__version__)
if torch.cuda.get_device_properties(0).total_memory / 2**30 < 12 and '4B' in MODEL:
    print('WARNING: under 12 GiB per GPU; Qwen3-4B in fp16 may not fit. Use GPU T4 x2.')

socket.create_connection(('huggingface.co', 443), timeout=10).close()
print('internet: OK')

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['PYTHONUNBUFFERED'] = '1'
if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
        print('HF_TOKEN loaded from Kaggle Secrets')
    except Exception as e:
        print(f'no HF_TOKEN secret ({type(e).__name__}): Qwen3-4B and UIT-VSFC still download; '
              'the NEU-ESC configuration will be skipped')

In [ ]:
# Code: the committed pipeline from GitHub. A re-run updates the code but keeps finished results
# (untracked run folders survive `git reset --hard`), so an interrupted session resumes.
if (WORK / '.git').exists():
    subprocess.run(['git', 'fetch', '--quiet', '--depth', '1', REPO_URL, BRANCH], cwd=WORK, check=True)
    subprocess.run(['git', 'reset', '--quiet', '--hard', 'FETCH_HEAD'], cwd=WORK, check=True)
else:
    if WORK.exists():
        shutil.rmtree(WORK)
    subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORK)], check=True)
sha = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=WORK, capture_output=True, text=True, check=True).stdout.strip()
print('commit', sha)

# The frozen inputs this run depends on must be in the clone.
import json

frozen = json.loads((WORK / 'results/studies/llm_reference/prompt_dev.json').read_text(encoding='utf-8'))
print('frozen prompt:', frozen['frozen_variant'], '| chosen with', frozen['model'], '@', frozen['revision'][:8])
for f in ('data/challenge/challenge_v1.csv', 'results/studies/challenge/predictions.csv',
          'results/studies/llm_reference/encoder_validation_preds.csv',
          'results/studies/external/neu_esc/predictions.csv'):
    assert (WORK / f).is_file(), f'missing in the clone: {f}'
print('frozen inputs present')

In [ ]:
# Dependencies: the package runs from the clone (PYTHONPATH), so pip never touches Kaggle's torch.
import importlib.metadata as md
import sys

from packaging.version import Version

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', *args], check=True)

if ON_KAGGLE:
    need = []
    for name, spec in (('typer', 'typer>=0.12'), ('pyyaml', 'pyyaml>=6.0'), ('pyarrow', 'pyarrow>=17')):
        try:
            md.version(name)
        except md.PackageNotFoundError:
            need.append(spec)
    if Version(md.version('transformers')) < Version('4.51'):   # Qwen3 support
        need.append('transformers>=4.51')
    if need:
        pip(*need)
    print('installed' if need else 'nothing to install', need)

ENV = dict(os.environ, PYTHONPATH=str(WORK / 'src'))

def cli(*args):
    print('$ vifeedback', ' '.join(args), flush=True)
    subprocess.run([sys.executable, '-W', 'ignore', '-m', 'vifeedback.cli', *args], cwd=WORK, env=ENV, check=True)

sys.path.insert(0, str(WORK / 'src'))
import transformers
import vifeedback

assert pathlib.Path(vifeedback.__file__).resolve().is_relative_to(WORK.resolve()), vifeedback.__file__
print('transformers', transformers.__version__, '| vifeedback from', pathlib.Path(vifeedback.__file__).parent)

In [ ]:
# Data: UIT-VSFC at the pinned revision, verified against the committed content hash.
cli('data', 'fetch')
# NEU-ESC (cycle3.yaml v6) is gated on the Hub: it needs the HF_TOKEN of an account that accepted its
# conditions. Without it only the NEU-ESC configuration is dropped; the declared Cycle 2 run goes on.
if any(c[0] == 'neu_esc' for c in CONFIGS):
    try:
        cli('data', 'fetch-external', '--name', 'neu_esc')   # verified against the pinned hashes
    except subprocess.CalledProcessError:
        CONFIGS = [c for c in CONFIGS if c[0] != 'neu_esc']
        print('WARNING: NEU-ESC could not be fetched (no HF_TOKEN secret, or its conditions are not '
              f'accepted on the dataset page). Continuing with {len(CONFIGS)} configurations.')


In [ ]:
# Sanity check before the long run: 24 challenge rows, zero-shot, on GPU 0, in this process.
# It stops here if fp16 overflows (non-finite scores) or the model answers at chance.
import gc

import numpy as np

from vifeedback.evaluation import challenge as CH
from vifeedback.evaluation import llm_reference as L

df = CH.load()                                   # raises if the file differs from its frozen hash
rows = df[df.scored].groupby('sentiment', group_keys=False).head(8)
scorer = L.HFScorer(MODEL, dtype=DTYPE)
print('revision', scorer.revision, '| single-token labels', scorer.single_token, '|', scorer._forward_kwargs())
r = L.run(scorer, rows.text.tolist(), frozen['frozen_variant'], batch_size=BATCH_SIZE, max_tokens=MAX_TOKENS)
acc = float((r['probs'].argmax(1) == rows.y.to_numpy().astype(int)).mean())
print(f'sanity: accuracy {acc:.2f} on {len(rows)} rows, {r["seconds_per_1k"]:.0f} s per 1k')
print('prefix cache:', scorer.prefix_cache_check)
assert np.isfinite(r['loglik']).all()
assert acc > SANITY_MIN_ACC, 'the model answers near chance: check the model id, dtype and transformers version'
del scorer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# The declared run: one worker per GPU. Each worker is ONE process that loads the model once and
# scores its queue in turn (`--configs`); a configuration that already has a summary is skipped, so
# a rerun resumes. Queues are balanced by estimated work: rows, x2 per NEU-ESC post (longer text),
# x1.3 for six-shot prompts (a longer cached prefix to attend to).
import threading
import time

OUT_DIR = WORK / 'results/studies/llm_reference' / L.slug(MODEL)
LOGS = WORK / 'h7_logs'
LOGS.mkdir(exist_ok=True)
ROWS = {'challenge': 305, 'validation': 1583, 'neu_esc': 6613}
PER_ROW = {'challenge': 1.0, 'validation': 1.0, 'neu_esc': 2.0}

def run_name(data, shots, seed):
    return f"{data}-{frozen['frozen_variant']}-k{shots}" + (f'-s{seed}' if shots else '')

def workload(c):
    return ROWS[c[0]] * PER_ROW[c[0]] * (1.3 if c[1] else 1.0)

def finished(c):
    return (OUT_DIR / run_name(*c) / 'summary.json').exists()

todo = [c for c in CONFIGS if not finished(c)]
gpus = os.environ.get('H7_GPUS', ','.join(str(i) for i in range(torch.cuda.device_count()))).split(',')
queues = [[] for _ in gpus]
for c in sorted(todo, key=workload, reverse=True):        # heaviest first, to the lightest queue
    min(queues, key=lambda q: sum(map(workload, q))).append(c)
for g, q in zip(gpus, queues):
    print(f'GPU {g}: {[run_name(*c) for c in q]}')
if len(todo) < len(CONFIGS):
    print(f'{len(CONFIGS) - len(todo)} configuration(s) already finished: skipped')

exit_codes = {}

def worker(gpu, queue):
    env = dict(ENV, CUDA_VISIBLE_DEVICES=gpu)
    spec = ','.join(f'{d}:{k}:{s}' for d, k, s in queue)
    with open(LOGS / f'gpu{gpu}.log', 'w', encoding='utf-8') as log:
        exit_codes[gpu] = subprocess.run(
            [sys.executable, '-W', 'ignore', '-m', 'vifeedback.cli', 'study', 'llm-reference',
             '--model', MODEL, '--configs', spec,
             '--batch-size', str(BATCH_SIZE), '--max-tokens', str(MAX_TOKENS), '--dtype', DTYPE],
            cwd=WORK, env=env, stdout=log, stderr=subprocess.STDOUT,
        ).returncode

t0 = time.time()
threads = [threading.Thread(target=worker, args=(g, q)) for g, q in zip(gpus, queues) if q]
for th in threads:
    th.start()
reported = set()
while any(th.is_alive() for th in threads):                # progress as each configuration lands
    time.sleep(20)
    for c in todo:
        if c not in reported and finished(c):
            reported.add(c)
            print(f'  done: {run_name(*c)} after {(time.time() - t0) / 60:.1f} min', flush=True)
for th in threads:
    th.join()

missing = [run_name(*c) for c in CONFIGS if not finished(c)]
for g, q in zip(gpus, queues):
    if q and (exit_codes.get(g) != 0 or any(run_name(*c) in missing for c in q)):
        print(f'--- GPU {g} exit {exit_codes.get(g)}; last lines of gpu{g}.log ---')
        print(''.join((LOGS / f'gpu{g}.log').read_text(encoding='utf-8', errors='replace').splitlines(True)[-25:]))
assert not missing, f'{len(missing)} configuration(s) missing: {missing}. Run the notebook again to resume.'
print(f'all {len(CONFIGS)} configurations finished ({(time.time() - t0) / 60:.1f} min in this session)')


In [ ]:
# Summary: LLM minus the CE encoder (the declared comparison), paired bootstrap over examples.
import pandas as pd

table = []
for data, shots, seed in CONFIGS:
    s = json.loads((OUT_DIR / run_name(data, shots, seed) / 'summary.json').read_text(encoding='utf-8'))
    n = s['vs_encoder']['ce']['neutral_f1']
    table.append({
        'data': data, 'shots': shots, 'seed': seed if shots else '',
        'macro-F1': round(s['macro_f1'], 4),
        'neutral F1': round(s['per_class']['neutral']['f1'], 3),
        'neutral P / R': f"{s['per_class']['neutral']['precision']:.2f} / {s['per_class']['neutral']['recall']:.2f}",
        'neutral F1 - CE [95% CI]': f"{n['observed_diff']:+.3f} [{n['ci_low']:+.3f}, {n['ci_high']:+.3f}]",
        'p': round(n['p_value'], 4),
        's per 1k': round(s['seconds_per_1k'], 1),
        'prefix cache': s.get('prefix_cache', {}).get('used'),
    })
print(pd.DataFrame(table).to_string(index=False))

In [ ]:
# Package: the whole results/ folder (a few MB), so `vifeedback results merge` finds registry.csv.
import zipfile

out = OUT_BASE / 'h7_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted((WORK / 'results').rglob('*')):
        if f.is_file() and '/local/' not in f.as_posix():
            z.write(f, f.relative_to(WORK / 'results'))
    for f in sorted(LOGS.glob('*.log')):
        z.write(f, pathlib.Path('studies/llm_reference') / L.slug(MODEL) / 'kaggle_logs' / f.name)
print(f'{out} ({out.stat().st_size / 1e6:.1f} MB)')
print('new run folders:', *sorted(p.name for p in OUT_DIR.iterdir() if p.is_dir()), sep='\n  ')
print('code commit:', sha)

## After the download

On the laptop, in the repository:

```
# extract h7_results.zip into kaggle_results/ (its own folder), then
vifeedback results merge kaggle_results --dry-run   # lists llm_reference/Qwen__Qwen3-4B/... as new
vifeedback results merge kaggle_results
vifeedback study h7-decide    # the H7 rule: Holm over Qwen3-4B and gpt-4o-mini (cycle3.yaml v6)
```

The merge only adds new run folders and never overwrites one. The H7 decision (Holm over Qwen3-4B
and gpt-4o-mini, `cycle2.yaml`) is applied on the laptop once both arms exist.